In [1]:
%load_ext autoreload
%autoreload 2

from dateutil import parser
from zoneinfo import ZoneInfo
from datetime import timezone, timedelta

from pynims.workflows import download_images_for_camera
from pynims.client import NIMSClient
from pynims.utils import convert_nims_image_name_to_utc_date

import dataretrieval.nwis as nwis

import pandas as pd
from IPython.display import display

In [2]:
# Get current run config files
import glob
import os 

config_paths = sorted(glob.glob('cameras/*/*/run_config.json'))

for cfg in config_paths:
    parts = cfg.split(os.sep)
    print(f"camera_id = '{parts[1]}'")
    print(f"run_name  = '{parts[2]}'")
    print()


camera_id = 'CA_Arroyo_DE_LA_Laguna_A_Corte_Madrid_nr_Pleasanton'
run_name  = 'event_2026-02-15'

camera_id = 'OK_Illinois_River_near_Moodys'
run_name  = 'event_2026-03-04'

camera_id = 'SC_Waccamaw_River_at_SC_22_below_Longs'
run_name  = 'event_2025-08-14'

camera_id = 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX'
run_name  = 'event_2026-03-16'

camera_id = 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet'
run_name  = 'event_2025-06-23'



In [10]:
# Set some initial parameters
site = '02110525'
camera_id = 'SC_Waccamaw_River_at_SC_22_below_Longs'
run_name = 'event_2025-08-14'
max_results = 2000  # None

run_dir = f'cameras/{camera_id}/{run_name}'
save_dir = f'{run_dir}/images'

start_all = '2025-08-14'  # local time
end_all = '2025-09-17'  # local time

start_event = '2025-08-14 15:30'  # local time
end_event = '2025-09-16 19:45'  # local time

use_event_times = True

allowable_im_data_time_diff = 60  # in seconds

# Load existing run config if available (for reproducibility)
import json, os
_config_path = config_paths[-1]
_existing_config = {}
if os.path.exists(_config_path):
    with open(_config_path) as f:
        _existing_config = json.load(f)
    print(f'Loaded existing config from {_config_path}')

    site = _existing_config.get('site')
    camera_id = _existing_config.get('camera_id')
    run_name = _existing_config.get('run_name')
    max_results = _existing_config.get('max_results')

    run_dir = f'cameras/{camera_id}/{run_name}'
    save_dir = f'{run_dir}/images'

    start_all = _existing_config.get('start_all')
    end_all = _existing_config.get('end_all')

    start_event = _existing_config.get('start_event')
    end_event = _existing_config.get('end_event')

    use_event_times = _existing_config.get('use_event_times')
    
    allowable_im_data_time_diff = _existing_config.get('allowable_im_data_time_diff')



Loaded existing config from cameras/WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet/event_2025-06-23/run_config.json


In [11]:
with NIMSClient() as client:
    tz = client.get_camera_attribute(camera_id, "tz")
print(f"==>> tz: {tz}")

if use_event_times:
    start = parser.parse(start_event)
    end = parser.parse(end_event)
else:
    start = parser.parse(start_all)
    end = parser.parse(end_all)


# Attach known timezone (if not already present)
if start.tzinfo is None:
    start = start.replace(tzinfo=ZoneInfo(tz))
if end.tzinfo is None:
    end = end.replace(tzinfo=ZoneInfo(tz))

# Convert to UTC
start = start.astimezone(timezone.utc)
print(f"==>> start (utc): {start}")
end = end.astimezone(timezone.utc)
print(f"==>> end (utc): {end}")

==>> tz: US/Central
==>> start (utc): 2025-06-23 23:00:00+00:00
==>> end (utc): 2025-06-26 15:00:00+00:00


In [12]:
with NIMSClient() as client:
    image_list = client.get_image_list(
        camera_id, start, end, recursive=None, max_results=max_results
    )
print(f"==>> image_list: {image_list}")
print(f"==>> len(image_list): {len(image_list)}")

==>> image_list: ['WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-23T23-00-02Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T00-00-03Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T01-00-02Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T02-00-02Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T03-00-03Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T05-00-06Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T06-00-02Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T07-00-03Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T08-00-07Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T09-00-03Z.jpg', 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T10-00-02Z.jpg', 'WI

In [14]:
keep_every_nth_image = _existing_config.get('keep_every_nth_image', 1) # set to 1 to keep all images
image_list = image_list[0::keep_every_nth_image]
print(f"==>> len(image_list) after keeping every {keep_every_nth_image}th image: {len(image_list)}")

==>> len(image_list) after keeping every 1th image: 62


In [15]:
download_images_for_camera(camera_id, start, end, max_results=max_results, save_dir=save_dir, image_list=image_list)

image # 1 of 62
WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-23T23-00-02Z.jpg already exists -- skipping
image # 2 of 62
image # 3 of 62
image # 4 of 62
image # 5 of 62
image # 6 of 62
image # 7 of 62
image # 8 of 62
image # 9 of 62
image # 10 of 62
image # 11 of 62
image # 12 of 62
image # 13 of 62
image # 14 of 62
image # 15 of 62
image # 16 of 62
image # 17 of 62
image # 18 of 62
image # 19 of 62
image # 20 of 62
image # 21 of 62
WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-24T20-00-16Z.jpg already exists -- skipping
image # 22 of 62
image # 23 of 62
image # 24 of 62
image # 25 of 62
image # 26 of 62
image # 27 of 62
image # 28 of 62
image # 29 of 62
image # 30 of 62
image # 31 of 62
image # 32 of 62
image # 33 of 62
image # 34 of 62
image # 35 of 62
image # 36 of 62
image # 37 of 62
image # 38 of 62
image # 39 of 62
image # 40 of 62
image # 41 of 62
WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet___2025-06-25T16-00-03Z.jpg 

In [16]:
image_times = [convert_nims_image_name_to_utc_date(image) for image in image_list]
print([dt.isoformat() for dt in image_times])
print(f"==>> len(image_times): {len(image_times)}")

['2025-06-23T23:00:02+00:00', '2025-06-24T00:00:03+00:00', '2025-06-24T01:00:02+00:00', '2025-06-24T02:00:02+00:00', '2025-06-24T03:00:03+00:00', '2025-06-24T05:00:06+00:00', '2025-06-24T06:00:02+00:00', '2025-06-24T07:00:03+00:00', '2025-06-24T08:00:07+00:00', '2025-06-24T09:00:03+00:00', '2025-06-24T10:00:02+00:00', '2025-06-24T11:00:03+00:00', '2025-06-24T12:00:07+00:00', '2025-06-24T13:00:03+00:00', '2025-06-24T14:00:02+00:00', '2025-06-24T15:00:02+00:00', '2025-06-24T16:00:07+00:00', '2025-06-24T17:00:08+00:00', '2025-06-24T18:00:02+00:00', '2025-06-24T19:00:02+00:00', '2025-06-24T20:00:16+00:00', '2025-06-24T21:00:03+00:00', '2025-06-24T22:00:04+00:00', '2025-06-24T23:00:03+00:00', '2025-06-25T00:00:04+00:00', '2025-06-25T01:00:08+00:00', '2025-06-25T02:00:08+00:00', '2025-06-25T03:00:02+00:00', '2025-06-25T04:00:05+00:00', '2025-06-25T05:00:02+00:00', '2025-06-25T06:00:05+00:00', '2025-06-25T07:00:04+00:00', '2025-06-25T08:00:02+00:00', '2025-06-25T09:00:14+00:00', '2025-06-25T1

In [17]:
# Data request requires start and end date in format YYYY-MM-DD
start_minus_one_day = (start - timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0).strftime("%Y-%m-%d")
end_plus_one_day = (end + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0).strftime("%Y-%m-%d")

# Get data
data_df = nwis.get_record(sites=site, service='iv', start=start_minus_one_day, end=end_plus_one_day)

# Filter data to our original range
mask = (data_df.index >= start) & (data_df.index <= end)
data_df = data_df.loc[mask]

display(data_df)

,site_no,00060,00060_cd,00065,00065_cd,63160,63160_cd
datetime,,,,,,,
2025-06-23 23:00:00+00:00,05433000,145.0,A,4.60,A,801.43,P
2025-06-23 23:15:00+00:00,05433000,145.0,A,4.60,A,801.43,P
2025-06-23 23:30:00+00:00,05433000,145.0,A,4.60,A,801.43,P
2025-06-23 23:45:00+00:00,05433000,144.0,A,4.60,A,801.43,P
2025-06-24 00:00:00+00:00,05433000,144.0,A,4.60,A,801.43,P
...,...,...,...,...,...,...,...
2025-06-26 14:00:00+00:00,05433000,322.0,A,6.54,A,803.37,P
2025-06-26 14:15:00+00:00,05433000,321.0,A,6.52,A,803.35,P
2025-06-26 14:30:00+00:00,05433000,320.0,A,6.52,A,803.35,P


In [18]:
image_df = pd.DataFrame({"image_times": image_times, "image_names": image_list})
display(image_df)

,image_times,image_names
0,2025-06-23 23:00:02+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...
1,2025-06-24 00:00:03+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...
2,2025-06-24 01:00:02+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...
3,2025-06-24 02:00:02+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...
4,2025-06-24 03:00:03+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...
...,...,...
57,2025-06-26 10:00:04+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...
58,2025-06-26 11:00:09+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...
59,2025-06-26 12:00:02+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...
60,2025-06-26 13:00:03+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...


In [19]:
merged = pd.merge_asof(
    image_df.sort_values("image_times"),
    data_df.reset_index().rename(columns={"datetime": "data_times"}),
    left_on="image_times",
    right_on="data_times",
    direction="nearest",
)

merged["time_diff_sec"] = (merged["image_times"] - merged["data_times"]).abs().dt.total_seconds()
filtered = merged[merged["time_diff_sec"] <= allowable_im_data_time_diff]

display(merged)

,image_times,image_names,data_times,site_no,00060,00060_cd,00065,00065_cd,63160,63160_cd,time_diff_sec
0,2025-06-23 23:00:02+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-23 23:00:00+00:00,05433000,145.0,A,4.60,A,801.43,P,2.0
1,2025-06-24 00:00:03+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-24 00:00:00+00:00,05433000,144.0,A,4.60,A,801.43,P,3.0
2,2025-06-24 01:00:02+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-24 01:00:00+00:00,05433000,145.0,A,4.60,A,801.43,P,2.0
3,2025-06-24 02:00:02+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-24 02:00:00+00:00,05433000,163.0,A,4.84,A,801.67,P,2.0
4,2025-06-24 03:00:03+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-24 03:00:00+00:00,05433000,201.0,A,5.27,A,802.10,P,3.0
...,...,...,...,...,...,...,...,...,...,...,...
57,2025-06-26 10:00:04+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-26 10:00:00+00:00,05433000,343.0,A,6.72,A,803.55,P,4.0
58,2025-06-26 11:00:09+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-26 11:00:00+00:00,05433000,337.0,A,6.67,A,803.50,P,9.0
59,2025-06-26 12:00:02+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-26 12:00:00+00:00,05433000,332.0,A,6.62,A,803.45,P,2.0
60,2025-06-26 13:00:03+00:00,WI_East_Branch_Pecatonica_River_near_Blanchard...,2025-06-26 13:00:00+00:00,05433000,327.0,A,6.58,A,803.41,P,3.0


In [20]:
merged.to_csv(f'{run_dir}/images_and_data.csv')

In [21]:
run_config = {
    "camera_id": camera_id,
    "site": site,
    "run_name": run_name,
    "start_all": start_all,
    "end_all": end_all,
    "start_event": start_event,
    "end_event": end_event,
    "use_event_times": use_event_times,
    "max_results": max_results,
    "keep_every_nth_image": keep_every_nth_image,
    "allowable_im_data_time_diff": allowable_im_data_time_diff,
}

os.makedirs(run_dir, exist_ok=True)
config_path = f'{run_dir}/run_config.json'
with open(config_path, 'w') as f:
    json.dump(run_config, f, indent=2)

print(f"Run config saved to {config_path}")

Run config saved to cameras/WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet/event_2025-06-23/run_config.json
